# EDA


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import os

In [2]:
path_to_training_file = os.path.join('datasets', 'training.csv')
path_to_validation_file = os.path.join('datasets', 'validation.csv')


df_training = pd.read_csv(path_to_training_file, sep='\t')
df_validation = pd.read_csv(path_to_validation_file, sep='\t')


In [3]:
columns_to_convert = ['ActualTOW','FLownPassengers','BagsCount','FlightBagsWeight']
for i in columns_to_convert:
    df_training[f'{i}']=pd.to_numeric(df_training[f'{i}'], errors='coerce')
val_columns_to_convert = ['FLownPassengers','BagsCount','FlightBagsWeight']
for i in val_columns_to_convert:
    df_validation[f'{i}']=pd.to_numeric(df_validation[f'{i}'], errors='coerce')

In [4]:

df = pd.concat([df_training,df_validation], axis=0)

## Numerical analysis

In [5]:
df.head()

,DepartureDate,DepartureYear,DepartureMonth,DepartureDay,FlightNumber,DepartureAirport,ArrivalAirport,Route,ActualFlightTime,ActualTotalFuel,ActualTOW,FLownPassengers,BagsCount,FlightBagsWeight
0,01/10/2016,2016,10,1,1145,MAN,SXF,MAN-SXF,91,3660,64016.0,175.0,61.0,440.0
1,01/10/2016,2016,10,1,1160,CTA,FCO,CTA-FCO,68,3280,66138.0,182.0,27.0,350.0
2,01/10/2016,2016,10,1,1183,LGW,SNN,LGW-SNN,64,2720,58447.0,127.0,NaN,NaN
3,01/10/2016,2016,10,1,1220,ATH,CHQ,ATH-CHQ,35,1682,60587.0,163.0,12.0,150.0
4,01/10/2016,2016,10,1,1225,CHQ,ATH,CHQ-ATH,34,1877,63090.0,178.0,37.0,490.0


In [6]:
print(df_training.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29731 entries, 0 to 29730
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DepartureDate     29731 non-null  object 
 1   DepartureYear     29731 non-null  int64  
 2   DepartureMonth    29731 non-null  int64  
 3   DepartureDay      29731 non-null  int64  
 4   FlightNumber      29731 non-null  int64  
 5   DepartureAirport  29731 non-null  object 
 6   ArrivalAirport    29731 non-null  object 
 7   Route             29731 non-null  object 
 8   ActualFlightTime  29731 non-null  int64  
 9   ActualTotalFuel   29731 non-null  int64  
 10  ActualTOW         29298 non-null  float64
 11  FLownPassengers   29636 non-null  float64
 12  BagsCount         27447 non-null  float64
 13  FlightBagsWeight  27253 non-null  float64
dtypes: float64(4), int64(6), object(4)
memory usage: 3.2+ MB
None


In [7]:
print(df_validation.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1878 entries, 0 to 1877
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DepartureDate     1878 non-null   object 
 1   DepartureYear     1878 non-null   int64  
 2   DepartureMonth    1878 non-null   int64  
 3   DepartureDay      1878 non-null   int64  
 4   FlightNumber      1878 non-null   int64  
 5   DepartureAirport  1878 non-null   object 
 6   ArrivalAirport    1878 non-null   object 
 7   Route             1878 non-null   object 
 8   ActualFlightTime  1878 non-null   int64  
 9   ActualTotalFuel   1878 non-null   int64  
 10  FLownPassengers   1869 non-null   float64
 11  BagsCount         1710 non-null   float64
 12  FlightBagsWeight  1697 non-null   float64
dtypes: float64(3), int64(6), object(4)
memory usage: 190.9+ KB
None


There are some null values in both training and validation datasets. Maybe the best method for filling out the missing values will be discovered later in the analysis. Let's take a look at the columns with 'object' data type:

In [8]:
print(df_training['DepartureDate'].unique())
date_train = go.Figure()
date_train.add_trace(
                go.Histogram(x=df_training['DepartureDate']
                     )
                        )
date_train.update_layout(title='Distribution of departure dates')


['01/10/2016' '02/10/2016' '03/10/2016' '04/10/2016' '05/10/2016'
 '06/10/2016' '07/10/2016' '08/10/2016' '09/10/2016' '10/10/2016'
 '11/10/2016' '12/10/2016' '13/10/2016' '14/10/2016' '15/10/2016']


In [9]:
print(df_validation['DepartureDate'].unique())

['01/11/2016']


Now, there's the first obstacle which is an outlying date of '01/11/2016'. In the provided training dataset all the entries come from 01/10/2016-15/10/2016 period. It might cause some problems later, when the data will be used to train a machine learning model. Especially because there is no variation in the validation dataset when it comes to this specific column. 

On the other side, the distribution of dates is fairly uniform, which is a good sign in this case.

In [10]:
print(f'Amount of unique departure airports in training: {len(df_training['DepartureAirport'].unique())}, amount of unique values in validation: {len(df_validation['DepartureAirport'].unique())}')
for i in df_validation['DepartureAirport'].values:
    if i not in df_training['DepartureAirport'].values:
        print(f"{i} not present in training set")

Amount of unique departure airports in training: 191, amount of unique values in validation: 170
OSL not present in training set
SZY not present in training set
LUX not present in training set
OSL not present in training set
OSL not present in training set
OSL not present in training set
LUX not present in training set
VDA not present in training set


It appears that despite being more rich in possible airport values, there are some locations that appear only in the validation dataset. Let's see if this problem is present in other columns.

In [11]:
print(f'Amount of unique arrival airports in training: {len(df_training['ArrivalAirport'].unique())}, amount of unique values in validation: {len(df_validation['ArrivalAirport'].unique())}')
for i in df_validation['ArrivalAirport'].values:
    if i not in df_training['ArrivalAirport'].values:
        print(f"{i} not present in training set")

Amount of unique arrival airports in training: 190, amount of unique values in validation: 170
OSL not present in training set
SZY not present in training set
LUX not present in training set
OSL not present in training set
OSL not present in training set
OSL not present in training set
LUX not present in training set
VDA not present in training set


The same problem happens with arrival airports. Now, let's take a look how it affects the "Route" column, which is a combination of previous columns.

In [12]:
print(f'Amount of unique routes in training: {len(df_training['Route'].unique())}, amount of unique values in validation: {len(df_validation['Route'].unique())}')
route_count=0
for i in df_validation['Route'].values:
    if i not in df_training['Route'].values:
        route_count += 1
print (f"Number of routes from validation not present in training set: {route_count}")

Amount of unique routes in training: 2667, amount of unique values in validation: 1383
Number of routes from validation not present in training set: 122


Well, in this case it is more efficient to change string values to numeric (longitude, latitude). However, this won't come without some information loss, because some flight routes for example need to avoid foribidden teritories, which results in increasing distance the plane has to beat. This problem is actually partialy resolved with the "AcualFlightTime" column, but it is just an example of possible loss of information.

Now, let's take a look at the numerical values in the dataset:

In [16]:
df_training.describe()

,DepartureYear,DepartureMonth,DepartureDay,FlightNumber,ActualFlightTime,ActualTotalFuel,ActualTOW,FLownPassengers,BagsCount,FlightBagsWeight
count,29731.0,29731.0,29731.000000,29731.000000,29731.000000,29731.000000,29298.000000,29636.000000,27447.000000,27253.000000
mean,2016.0,10.0,7.987723,4814.133934,110.069221,4782.483536,65031.246638,167.268626,45.417022,573.618464
std,0.0,0.0,4.313351,3062.612334,52.457470,2069.865460,3339.385728,20.177527,32.797217,376.524581
min,2016.0,10.0,1.000000,1.000000,2.000000,600.000000,42991.000000,4.000000,1.000000,3.000000
25%,2016.0,10.0,4.000000,2087.000000,73.000000,3210.000000,63127.000000,163.000000,23.000000,300.000000
50%,2016.0,10.0,8.000000,4765.000000,105.000000,4530.000000,65122.000000,173.000000,39.000000,490.000000
75%,2016.0,10.0,12.000000,7496.000000,140.000000,6010.000000,67154.000000,179.000000,62.000000,780.000000
max,2016.0,10.0,15.000000,9998.000000,1504.000000,17890.000000,74283.000000,189.000000,1180.000000,10530.000000


In [17]:
df_validation.describe()

,DepartureYear,DepartureMonth,DepartureDay,FlightNumber,ActualFlightTime,ActualTotalFuel,FLownPassengers,BagsCount,FlightBagsWeight
count,1878.0,1878.0,1878.0,1878.000000,1878.000000,1878.000000,1869.000000,1710.000000,1697.000000
mean,2016.0,11.0,1.0,4673.227370,110.446219,4709.011715,166.103264,36.250292,446.149087
std,0.0,0.0,0.0,3069.845977,51.150847,2071.596772,26.391906,23.799249,275.057285
min,2016.0,11.0,1.0,11.000000,7.000000,1100.000000,6.000000,2.000000,20.000000
25%,2016.0,11.0,1.0,1995.250000,72.000000,3200.000000,163.000000,19.000000,250.000000
50%,2016.0,11.0,1.0,4566.500000,104.000000,4420.500000,174.000000,31.000000,390.000000
75%,2016.0,11.0,1.0,7452.750000,140.000000,5895.500000,180.000000,47.000000,590.000000
max,2016.0,11.0,1.0,9995.000000,318.000000,13040.000000,189.000000,260.000000,1700.000000


I will skip analysing the dates, because I already gave them some attention earlier in the string values analysis and there is not much more to conclude from the table above.

Flight number -  a value used only for identification, but it might provide some important insights, depending on the priciples used when determining the number. When researching the rules used by Ryanair I've found out, that the number of the flight is determined by some algorithm, maybe some form of clustering will provide some informations. 

Now, let's take a look at the graphs of the rest of the columns:

In [ ]:
test